# Multi-Agent Meal Planner Workflow

This notebook mirrors the backend orchestration used by the FastAPI app. It is designed for local exploration, debugging, and demos before changes are promoted into the live API.

The workflow follows the same sequence as `/generate-meal-plan`:

1. Capture user input and location context.
2. Load the user profile and dietary constraints.
3. Run the meal definition agent.
4. Run the nutrition agent with USDA, FatSecret, and local fallbacks.
5. Run the supermarket agent.
6. Aggregate the response into the API-compatible payload.

Secrets are loaded from `backend/.env`; no API keys are stored in this notebook.

## 1. Environment Setup

In [ ]:
from pathlib import Path
import json
import sys
from datetime import UTC, datetime
from uuid import uuid4

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

BACKEND_DIR = PROJECT_ROOT / "backend"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from backend.config import AppSettings
from backend.meal_definition_agent import MealDefinitionAgent
from backend.nutrition_agent import NutritionAgent
from backend.storage import MealPlanRepository, UserProfileRepository
from backend.supermarket_agent import SupermarketAgent

settings = AppSettings.from_env()

service_status = {
    "gemini_configured": bool(settings.gemini_api_key),
    "usda_configured": bool(settings.usda_api_key),
    "fatsecret_configured": bool(settings.fatsecret_client_id and settings.fatsecret_client_secret),
    "maps_configured": bool(settings.maps_api_key),
    "inventory_configured": bool(settings.inventory_api_key),
}
service_status

## 2. Define the Multimodal Request

The current product primarily accepts text input plus structured user and location context. The `optional_image_note` field is included as a placeholder for a future visual modality, such as a food photo or pantry image.

In [ ]:
request_context = {
    "user_id": "user_123",
    "craving": "asian noodle",
    "location": "Earlwood, NSW",
    "optional_image_note": None,
}

request_context

## 3. Initialize Agents and Storage

In [ ]:
user_profiles = UserProfileRepository(settings.data_dir)
meal_history = MealPlanRepository(settings.data_dir)

meal_agent = MealDefinitionAgent(
    db_connection=user_profiles,
    gemini_api_key=settings.gemini_api_key,
)

nutrition_agent = NutritionAgent(
    usda_api_key=settings.usda_api_key,
    fatsecret_client_id=settings.fatsecret_client_id,
    fatsecret_client_secret=settings.fatsecret_client_secret,
)

supermarket_agent = SupermarketAgent(
    maps_api_key=settings.maps_api_key,
    inventory_api_key=settings.inventory_api_key,
)

"Agents initialized"

## 4. User Profile and Calorie Context

In [ ]:
user_profile = user_profiles.fetch_user_profile(request_context["user_id"])
caloric_target = meal_agent.calculate_bmr(
    age=user_profile["age"],
    gender=user_profile["gender"],
    weight_kg=user_profile["weight"],
    height_cm=user_profile["height"],
    activity_multiplier=user_profile["workout_level"],
)

{
    "user_profile": user_profile,
    "caloric_target": caloric_target,
}

## 5. Agent 1: Meal Definition

This agent converts the craving into a structured meal and raw ingredient list. If Gemini is unavailable or quota-limited, it returns a deterministic fallback with metadata explaining why.

In [ ]:
meal_payload = meal_agent.generate_meal_payload(
    craving=request_context["craving"],
    user_id=request_context["user_id"],
)

meal_payload.model_dump()

## 6. Agent 2: Nutrition

The nutrition provider order is USDA, then FatSecret, then local reference/category estimates. Each item includes a `data_source` and confidence score.

In [ ]:
nutrition_payload = nutrition_agent.calculate_meal_macros(
    ingredients=meal_payload.meal_definition.ingredients,
)

nutrition_payload.model_dump()

## 7. Agent 3: Supermarket Mapping

The supermarket agent maps generic ingredients to grocery products and estimates total cost.

In [ ]:
shopping_payload = supermarket_agent.generate_shopping_list(
    ingredients=meal_payload.meal_definition.ingredients,
    user_location=request_context["location"],
)

shopping_payload.model_dump()

## 8. Aggregate API-Compatible Response

In [ ]:
aggregated_response = {
    "status": "success",
    "request_id": str(uuid4()),
    "generated_at": datetime.now(UTC).isoformat(),
    "request": {
        "user_id": request_context["user_id"],
        "craving": request_context["craving"],
        "location": request_context["location"],
    },
    "meal_plan": meal_payload.model_dump(),
    "nutrition": nutrition_payload.model_dump(),
    "shopping_list": shopping_payload.model_dump(),
}

print(json.dumps(aggregated_response, indent=2))

## 9. Optional: Save to Local Meal History

Uncomment the final line if you want this notebook run to be stored in `database/meal_history.json`, the same local history file used by the API.

In [ ]:
# meal_history.save(aggregated_response)
# "Saved to local meal history"

## 10. Optional: Call the Live FastAPI Endpoint

Run the backend first:

```powershell
venv\Scripts\python.exe -m uvicorn backend.main:app --host 0.0.0.0 --port 8000 --reload
```

In [ ]:
import urllib.error
import urllib.request

api_payload = json.dumps({
    "user_id": request_context["user_id"],
    "craving": request_context["craving"],
    "location": request_context["location"],
}).encode("utf-8")

api_request = urllib.request.Request(
    "http://localhost:8000/generate-meal-plan",
    data=api_payload,
    headers={"Content-Type": "application/json"},
    method="POST",
)

try:
    with urllib.request.urlopen(api_request, timeout=60) as response:
        live_response = json.loads(response.read().decode("utf-8"))
    print(json.dumps(live_response, indent=2))
except urllib.error.URLError as exc:
    print(f"Live API call skipped or failed: {exc}")